In [44]:
import numpy as np
import pandas as pd

# Load Data
df = pd.read_csv('laptop_data.csv')

# ---------------------- Basic Cleaning ----------------------
df.drop(columns=['Unnamed: 0'], inplace=True, errors='ignore')

df['Ram'] = df['Ram'].str.replace('GB', '', regex=False).astype('int32')
df['Weight'] = df['Weight'].str.replace('kg', '', regex=False).astype('float32')

# ---------------------- Feature Engineering ----------------------

# Touchscreen & IPS
df['Touchscreen'] = df['ScreenResolution'].apply(lambda x: 1 if 'Touchscreen' in x else 0)
df['Ips'] = df['ScreenResolution'].apply(lambda x: 1 if 'IPS' in x else 0)

# Resolution
new = df['ScreenResolution'].str.split('x', n=1, expand=True)
df['X_res'] = new[0]
df['Y_res'] = new[1]

df['X_res'] = df['X_res'].str.replace(',', '').str.findall(r'(\d+\.?\d+)').apply(lambda x: x[0]).astype(int)
df['Y_res'] = df['Y_res'].astype(int)

# PPI
df['ppi'] = (((df['X_res']**2 + df['Y_res']**2)**0.5) / df['Inches']).astype(float)

df.drop(columns=['ScreenResolution', 'Inches', 'X_res', 'Y_res'], inplace=True)

# CPU
df['Cpu Name'] = df['Cpu'].apply(lambda x: " ".join(x.split()[0:3]))

def fetch_processor(text):
    if text in ['Intel Core i7', 'Intel Core i5', 'Intel Core i3']:
        return text
    elif text.split()[0] == 'Intel':
        return 'Other Intel Processor'
    else:
        return 'AMD Processor'

df['Cpu brand'] = df['Cpu Name'].apply(fetch_processor)
df.drop(columns=['Cpu', 'Cpu Name'], inplace=True)

# ---------------------- Memory ----------------------
df['Memory'] = df['Memory'].astype(str).replace('\.0', '', regex=True)
df['Memory'] = df['Memory'].str.replace('GB', '')
df['Memory'] = df['Memory'].str.replace('TB', '000')

new = df['Memory'].str.split("+", n=1, expand=True)

df['first'] = new[0].str.strip()
df['second'] = new[1]

# Layer flags
for col in ['HDD', 'SSD']:
    df[f'Layer1{col}'] = df['first'].apply(lambda x: 1 if col in x else 0)
    df[f'Layer2{col}'] = df['second'].apply(lambda x: 1 if isinstance(x, str) and col in x else 0)

df['first'] = df['first'].str.replace(r'\D', '', regex=True).astype(int)
df['second'] = df['second'].fillna('0').str.replace(r'\D', '', regex=True).astype(int)

# Final storage
df['HDD'] = df['first'] * df['Layer1HDD'] + df['second'] * df['Layer2HDD']
df['SSD'] = df['first'] * df['Layer1SSD'] + df['second'] * df['Layer2SSD']

# Drop temp columns safely
df.drop(
    columns=[col for col in df.columns if 'Layer' in col] + ['first', 'second', 'Memory'],
    inplace=True,
    errors='ignore'
)

# ---------------------- GPU ----------------------
df['Gpu brand'] = df['Gpu'].apply(lambda x: x.split()[0])
df = df[df['Gpu brand'] != 'ARM']
df.drop(columns=['Gpu'], inplace=True)

# ---------------------- OS ----------------------
def cat_os(inp):
    if inp in ['Windows 10', 'Windows 7', 'Windows 10 S']:
        return 'Windows'
    elif inp in ['macOS', 'Mac OS X']:
        return 'Mac'
    else:
        return 'Others/Linux'

df['os'] = df['OpSys'].apply(cat_os)
df.drop(columns=['OpSys'], inplace=True)

# ---------------------- Modeling ----------------------
X = df.drop(columns=['Price'])
y = np.log(df['Price'])

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=2)

# Preprocessing
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

categorical_cols = ['Company', 'TypeName', 'Cpu brand', 'Gpu brand', 'os']

preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_cols)
], remainder='passthrough')

# Models
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor, VotingRegressor
from xgboost import XGBRegressor

rf = RandomForestRegressor(n_estimators=350, max_depth=15, max_features=0.75, random_state=3)
gbdt = GradientBoostingRegressor(n_estimators=100)
et = ExtraTreesRegressor(n_estimators=100, max_depth=15, random_state=3)
xgb = XGBRegressor(n_estimators=50, max_depth=5, learning_rate=0.3, verbosity=0)

model = VotingRegressor([
    ('rf', rf),
    ('gbdt', gbdt),
    ('et', et),
    ('xgb', xgb)
])

pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('scaler', StandardScaler()),
    ('model', model)
])

# Train
pipe.fit(X_train, y_train)

# Predict
y_pred = pipe.predict(X_test)

# Evaluation
print("R2 Score:", r2_score(y_test, y_pred))
print("MAE:", mean_absolute_error(y_test, y_pred))

# ---------------------- Save Model ----------------------
import pickle
pickle.dump(pipe, open('pipe.pkl', 'wb'))

print("✅ Model saved successfully!")

R2 Score: 0.8909531346032774
MAE: 0.15549282020160857
✅ Model saved successfully!
